# Named Entity Recognition (NER) with Transformers

This notebook demonstrates how to extract and classify named entities from text using pre-trained transformer models.

## What is Named Entity Recognition?

NER is the task of identifying and classifying named entities in text into predefined categories such as:
- **PER**: Person names
- **ORG**: Organizations
- **LOC**: Locations
- **MISC**: Miscellaneous entities

NER is crucial for information extraction, content analysis, and building knowledge graphs.

## Setup

In [ ]:
from transformers import pipeline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("Libraries imported successfully!")

## Load Pre-trained NER Model

We'll use a BERT model fine-tuned on the CoNLL-2003 NER dataset.

In [ ]:
# Load NER pipeline with aggregation strategy to combine sub-tokens
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

print("NER model loaded successfully!")

## Single Text Analysis

Let's extract entities from a single text:

In [ ]:
# Example text
text = "Apple Inc. was founded by Steve Jobs in Cupertino, California."

# Extract entities
entities = ner(text)

print(f"Text: {text}\n")
print(f"Found {len(entities)} entities:\n")

for entity in entities:
    print(f"  Entity: '{entity['word']}'")
    print(f"  Type: {entity['entity_group']}")
    print(f"  Confidence: {entity['score']:.4f}")
    print(f"  Position: {entity['start']}-{entity['end']}")
    print()

## Visualize Entity Positions

Let's create a visual representation of where entities appear in the text:

In [ ]:
# Color map for entity types
color_map = {
    'PER': 'lightblue',
    'ORG': 'lightgreen',
    'LOC': 'lightsalmon',
    'MISC': 'lightgray'
}

# Create visualization
fig, ax = plt.subplots(figsize=(14, 4))

# Display text
ax.text(0, 0.5, text, fontsize=12, va='center')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, 1)
ax.axis('off')

# Highlight entities
text_len = len(text)
for entity in entities:
    start_ratio = entity['start'] / text_len
    end_ratio = entity['end'] / text_len
    width = end_ratio - start_ratio
    
    color = color_map.get(entity['entity_group'], 'lightgray')
    ax.add_patch(plt.Rectangle((start_ratio, 0.35), width, 0.3, 
                                facecolor=color, alpha=0.3, edgecolor='black'))
    
    # Add label
    ax.text(start_ratio + width/2, 0.15, entity['entity_group'], 
            ha='center', fontsize=9, fontweight='bold')

plt.title('Entity Positions in Text', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Legend
print("\nEntity Type Legend:")
for entity_type, color in color_map.items():
    print(f"  {entity_type}: {color}")

## Batch Processing

Let's extract entities from multiple texts:

In [ ]:
# Multiple example texts
texts = [
    "Apple Inc. was founded by Steve Jobs in Cupertino, California.",
    "Barack Obama was the 44th President of the United States.",
    "Google announced a new AI research lab in London.",
    "The Eiffel Tower in Paris attracts millions of visitors.",
    "Microsoft CEO Satya Nadella spoke at the conference in Seattle.",
]

# Extract entities from all texts
all_entities = []
for i, text in enumerate(texts):
    entities = ner(text)
    for entity in entities:
        all_entities.append({
            'text_id': i,
            'entity': entity['word'],
            'type': entity['entity_group'],
            'confidence': entity['score']
        })

# Create DataFrame
df = pd.DataFrame(all_entities)
print(df.to_string(index=False))

## Entity Statistics and Visualization

In [ ]:
# Entity type distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Entity type counts
type_counts = df['type'].value_counts()
type_counts.plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Entity Type Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Entity Type')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=0)

# 2. Average confidence by type
avg_conf = df.groupby('type')['confidence'].mean().sort_values(ascending=False)
avg_conf.plot(kind='bar', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Average Confidence by Entity Type', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Entity Type')
axes[0, 1].set_ylabel('Average Confidence')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. Confidence distribution
axes[1, 0].hist(df['confidence'], bins=20, color='teal', edgecolor='black')
axes[1, 0].set_title('Confidence Score Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Confidence Score')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(df['confidence'].mean(), color='red', linestyle='--', 
                   label=f'Mean: {df["confidence"].mean():.3f}')
axes[1, 0].legend()

# 4. Entities per text
entities_per_text = df.groupby('text_id').size()
entities_per_text.plot(kind='bar', ax=axes[1, 1], color='purple')
axes[1, 1].set_title('Number of Entities per Text', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Text ID')
axes[1, 1].set_ylabel('Entity Count')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSummary Statistics:")
print(f"Total entities found: {len(df)}")
print(f"Average confidence: {df['confidence'].mean():.4f}")
print(f"Min confidence: {df['confidence'].min():.4f}")
print(f"Max confidence: {df['confidence'].max():.4f}")

## Entity Co-occurrence

Let's see which entity types appear together in texts:

In [ ]:
# Create co-occurrence matrix
from itertools import combinations

cooccurrence = {}
for text_id in df['text_id'].unique():
    types = df[df['text_id'] == text_id]['type'].unique()
    for type1, type2 in combinations(sorted(types), 2):
        pair = (type1, type2)
        cooccurrence[pair] = cooccurrence.get(pair, 0) + 1

if cooccurrence:
    print("Entity Type Co-occurrences:")
    for pair, count in sorted(cooccurrence.items(), key=lambda x: x[1], reverse=True):
        print(f"  {pair[0]} + {pair[1]}: {count} times")
else:
    print("No co-occurrences found (each text has only one entity type)")

## Interactive: Analyze Your Own Text!

In [ ]:
# Try your own text here!
your_text = "Elon Musk founded Tesla and SpaceX in California and later moved to Texas."

# Extract entities
entities = ner(your_text)

print(f"Your text: {your_text}\n")
print(f"Found {len(entities)} entities:\n")

# Create DataFrame
if entities:
    entity_df = pd.DataFrame([{
        'Entity': e['word'],
        'Type': e['entity_group'],
        'Confidence': f"{e['score']:.4f}",
        'Position': f"{e['start']}-{e['end']}"
    } for e in entities])
    print(entity_df.to_string(index=False))
    
    # Visualize
    plt.figure(figsize=(10, 6))
    type_counts = entity_df['Type'].value_counts()
    colors = [color_map.get(t, 'lightgray') for t in type_counts.index]
    type_counts.plot(kind='bar', color=colors, edgecolor='black')
    plt.title('Entity Types in Your Text', fontsize=14, fontweight='bold')
    plt.xlabel('Entity Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No entities found.")

## Key Takeaways

1. **Pre-trained NER models** are highly effective for entity extraction
2. **Aggregation strategy** is important for combining sub-word tokens
3. **Confidence scores** help identify uncertain predictions
4. **Context awareness**: Models understand "Apple" as organization vs. fruit
5. **Real-world applications**:
   - Information extraction from documents
   - Building knowledge graphs
   - Content categorization and tagging
   - Privacy protection (PII detection)
   - Question answering systems

## Limitations

- Limited to predefined entity types
- May struggle with rare or domain-specific entities
- Ambiguous entities can be challenging
- Performance depends on training data distribution

## Next Steps

- Fine-tune on domain-specific entities
- Add custom entity types
- Implement entity linking to knowledge bases
- Build entity relationship extraction
- Compare with spaCy and other NER tools